# DPR (2020)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
DPR = Dense Passage Retrieval Model

DPR — это реализация Dense Retrieval модели, предложенная командой Meta (тогда еще Facebook AI Research). Она использует два специализированных BERT-энкодера для получения плотных векторных представлений запросов и документов, а также **dot-product** в качестве меры схожести. Энкодеры обучаются с помощью **contrastive loss** на тщательно отобранных данных, включающих **Hard Negative** примеры.

### Идея
К моменту появления DPR уже существовали попытки использовать нейронные сети для Dense Retrieval, но они часто страдали от неоптимального обучения или архитектуры. Основная идея DPR заключалась в том, чтобы систематизировать лучший опыт в области нейронного Dense Retrieval и создать надежный, готовый к использованию фреймворк. Это было достигнуто за счет:
1.  Использования мощных предобученных трансформерных моделей (BERT).
2.  Применения двух *отдельных* энкодеров для запросов и документов, что позволяет им специализироваться на своих задачах.
3.  Разработки методологии обучения с **Hard Negative** примерами, которые значительно улучшают способность модели различать релевантные и нерелевантные документы.

### Постановка задачи
Задача **Passage Retrieval** состоит в следующем: дана база текстовых документов (пассажей) ${d_i} \in D$ и запрос $Q$. Необходимо найти и извлечь top-K документов, наиболее релевантных запросу $Q$. Релевантность определяется таким образом, чтобы помочь в решении конечных задач, например, Open-Domain Question Answering (QA).

### Предшествующие методы
До появления DPR существовали различные подходы к информационному поиску, каждый со своими особенностями:

*   **Sparse Retrieval (например, BM25):** Этот классический подход (который существует с 1990-х годов) основан на подсчете частоты слов и их инвертированной частоты в документах и запросе. Он очень эффективен с точки зрения скорости и простоты, но не способен улавливать семантическую схожесть или синонимию, поскольку работает на уровне токенов, а не значений.
*   **Word2Vec (2013):** Хоть и не является методом для Retrieval напрямую, Word2Vec был одним из первых, кто представил концепцию **word embeddings** — плотных векторных представлений слов. Он использовал неглубокие нейронные сети для обучения, но не был разработан для кодирования целых пассажей или запросов для Retrieval.
*   **DSSM (Deep Structured Semantic Model, 2013):** Одна из первых **двухбашенных** нейронных моделей для Retrieval. Она использовала отдельные сети для запроса и документа, пропуская текст через сверточные слои, но основной проблемой было использование **sparse bag-of-words** представлений на входе, что игнорировало порядок слов и тонкие контекстные связи. Кроме того, DSSM часто обучался на кликовых данных, что не всегда коррелирует с истинной семантической релевантностью.
*   **DrQA (2017):** Эта модель для Question Answering использовала **двухбашенную** архитектуру с LSTM-сетями для кодирования вопросов и документов. Модель обучалась **self-supervised** методом, например, с помощью **Inverse Cloze Task**, где модель предсказывала соседние предложения на основе контекста. Хотя это улучшало семантические представления, оно не было напрямую оптимизировано для задачи релевантности в Retrieval.
*   **OrQA (2019):** Предшественник DPR, который также использовал **двухбашенную** архитектуру на основе BERT. Однако OrQA в основном обучалась в **self-supervised** манере на задаче Masked Language Modeling (MLM), что, как и в случае с DrQA, не было прямой оптимизацией для задачи релевантности "запрос-документ". Это делало её менее эффективной в захвате тонких аспектов релевантности по сравнению с DPR.

Основное архитектурное отличие DPR от многих предшественников заключалось в целенаправленном использовании **двух отдельных BERT-энкодеров** (один для запроса, один для пассажа) и их **прямой оптимизации на задаче релевантности** через контрастивное обучение с использованием **Hard Negative** примеров.

### Архитектура
Архитектура DPR представляет собой **двухбашенную модель** (two-tower model) со следующими ключевыми компонентами:

1.  **Question Encoder ($E_Q$):** Отдельный BERT-энкодер, предназначенный для кодирования запросов $Q$. Он принимает на вход текст запроса и выводит его плотное векторное представление (эмбеддинг). Обычно для этого используется вектор, соответствующий [CLS] токену.
2.  **Passage Encoder ($E_P$):** Другой, независимый BERT-энкодер, предназначенный для кодирования документов (пассажей) $D$. Он принимает на вход текст пассажа и также выводит его плотное векторное представление (эмбеддинг [CLS] токена).
3.  **Метрика релевантности:** Оценка релевантности между запросом $Q$ и пассажем $D$ вычисляется как **dot-product** (скалярное произведение) их эмбеддингов: $score(Q, D) = E_Q(Q) \cdot E_P(D)$.

Важно, что энкодеры $E_Q$ и $E_P$ являются *разными* моделями BERT. Это позволяет им специализироваться на различных аспектах кодирования: запросы обычно короткие и требуют захвата основной интенции, в то время как пассажи длиннее и требуют более полного понимания их содержания.

### Алгоритм обучения
Обучение DPR происходит с использованием **contrastive learning** (контрастивного обучения), которое направлено на то, чтобы эмбеддинги релевантных пар (запрос, позитивный документ) были близки друг к другу, а нерелевантных пар (запрос, негативный документ) — далеки.

1.  **Формирование обучающего батча:**
    *   Для каждого запроса $Q$, имеющегося в батче, выбирается один **позитивный документ** $D^+$, который был аннотирован как релевантный (например, документ, содержащий ответ на вопрос).
    *   Выбирается набор **негативных документов** $D^-$. Это критически важный шаг:
        *   **In-batch Negatives:** Все остальные $D^+$ в том же обучающем батче, которые не являются позитивным для текущего $Q$, автоматически считаются негативными для этого $Q$. Это эффективный способ получения негативных примеров.
        *   **Hard Negatives:** Для каждого $Q$ также отбираются "сложные" негативные примеры. Это документы, которые BM25 (или другая сильная Sparse Retrieval модель) считает релевантными, но которые на самом деле не содержат ответа или не являются истинно релевантными. Эти примеры помогают модели учиться различать тонкие семантические различия. DPR также экспериментирует с **DPR-trained Hard Negatives**, где негативы подбираются уже обученной DPR-моделью, что еще больше усложняет задачу.
2.  **Вычисление скоров релевантности:** Для каждого запроса $Q$ из батча вычисляются скоры релевантности со всеми позитивными и негативными документами в батче с помощью dot-product: $s_i = E_Q(Q) \cdot E_P(D_i)$.
3.  **Нормализация и вероятность:** Полученные скоры преобразуются в вероятности с помощью функции **softmax**: $P(D_i | Q) = \frac{e^{s_i}}{\sum_{j=1}^{N} e^{s_j}}$, где $N$ — общее количество документов (позитивных и негативных) в батче для данного запроса.
4.  **Функция потерь:** Используется **Negative Log-Likelihood (NLL)** loss. Цель — максимизировать вероятность позитивного документа: $L = -\sum_{Q} \log P(D^+ | Q)$.
    *   *Почему не Triplet Loss?* Хотя Triplet Loss является стандартным для контрастивного обучения, NLL с in-batch negatives и hard negatives часто оказывается более эффективным. NLL оптимизирует относительные вероятности всех документов в батче, заставляя позитивный документ быть ближе к запросу, чем *все* негативные, а не только один "сложный" негативный, как в Triplet Loss.

### Индексирование
После обучения Passage Encoder $E_P$ используется для построения поискового индекса:

1.  **Генерация эмбеддингов:** Все документы (пассажи) из коллекции $D$ пропускаются через обученный $E_P$ для получения их плотных эмбеддингов.
2.  **Хранение в ANN-индексе:** Полученные эмбеддинги хранятся в специальной структуре данных, такой как **ANN (Approximate Nearest Neighbor)** индекс. Примеры таких индексов включают FAISS (Facebook AI Similarity Search), Annoy, HNSW. Эти индексы позволяют быстро находить ближайших соседей к заданному векторному запросу, даже в очень больших коллекциях документов.

### Инференс
Процесс поиска документов для нового запроса:

1.  **Кодирование запроса:** Входящий запрос $Q$ пропускается через обученный Question Encoder $E_Q$ для получения его эмбеддинга: $e_Q = E_Q(Q)$.
2.  **Поиск в индексе:** Эмбеддинг $e_Q$ используется для выполнения поиска ближайших соседей в заранее построенном ANN-индексе.
3.  **Извлечение top-K:** ANN-индекс возвращает top-K документов (точнее, их эмбеддинги), которые имеют наибольшее сходство (по dot-product) с эмбеддингом запроса.
4.  **Дальнейшая обработка (опционально):** Полученные top-K документы могут быть переданы на дальнейшую обработку, например, в **Reader** модель (для извлечения точного ответа в QA-системах) или **Re-ranker** модель (для более точной сортировки документов с помощью более сложной, но медленной модели).

### Результаты
DPR продемонстрировал значительные улучшения в эффективности Dense Retrieval по сравнению с существовавшими на тот момент методами:

*   **Значительное превосходство над Sparse Retrieval:** На нескольких датасетах для Open-Domain QA, таких как Natural Questions и TriviaQA, DPR показал улучшение в метрике Top-20 Accuracy на 10-20 процентных пунктов по сравнению с сильной baseline Sparse Retrieval моделью, такой как BM25. Например, на Natural Questions точность возросла с ~60% (для BM25) до ~80% (для DPR).
*   **Эффективность Hard Negative сэмплинга:** Исследование показало, что использование **Hard Negative** примеров является критически важным для достижения высокой производительности. Обучение без них или только на случайных негативах приводит к значительному падению качества Retrieval.
*   **SOTA в Open-Domain QA:** В сочетании с мощным Reader'ом, DPR позволил достичь state-of-the-art результатов в Open-Domain QA, значительно упростив и улучшив этап Retrieval по сравнению с предыдущими подходами.

## 📝 Критический анализ

```markdown
# DPR (2020)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
DPR = Dense Passage Retrieval Model

DPR — реализация Dense Retrieval модели от Meta, использующая два BERT-энкодера для получения векторных представлений запросов и документов. Энкодеры обучаются с помощью **contrastive loss** на данных с **Hard Negative** примерами.

### Идея
DPR систематизирует опыт в нейронном Dense Retrieval, создавая надежный фреймворк. Это достигается использованием BERT, двумя отдельными энкодерами для запросов и документов и обучением с **Hard Negative** примерами.

### Задача
Задача **Passage Retrieval**: из базы текстовых документов ${d_i} \in D$ и запроса $Q$ извлечь top-K релевантных документов. Релевантность помогает решать задачи, например, Open-Domain QA.

### Предшественники
- **Sparse Retrieval (BM25):** Эффективен, но не улавливает семантику.
- **Word2Vec (2013):** Ввел **word embeddings**, но не для Retrieval.
- **DSSM (2013):** Двухбашенная модель, но с **sparse** представлениями.
- **DrQA (2017):** Использовал LSTM, обучался **self-supervised**.
- **OrQA (2019):** Двухбашенная BERT, обучалась на MLM, менее эффективна.

DPR отличается использованием **двух BERT-энкодеров** и оптимизацией на задаче релевантности через контрастивное обучение.

### Архитектура
- **Question Encoder ($E_Q$):** Кодирует запросы в векторное представление.
- **Passage Encoder ($E_P$):** Кодирует документы.
- **Метрика релевантности:** **Dot-product** эмбеддингов.

Энкодеры $E_Q$ и $E_P$ — разные модели BERT, специализирующиеся на своих задачах.

### Обучение
Обучение с **contrastive learning**:
1. **Формирование батча:** Для каждого $Q$ выбирается $D^+$ и набор $D^-$.
2. **Скоры релевантности:** Вычисляются для всех документов.
3. **Нормализация:** **Softmax** для вероятностей.
4. **Функция потерь:** **Negative Log-Likelihood (NLL)**.

### Индексирование
1. **Генерация эмбеддингов:** Документы пропускаются через $E_P$.
2. **Хранение в ANN-индексе:** Используются структуры, такие как FAISS.

### Инференс
1. **Кодирование запроса:** $Q$ через $E_Q$.
2. **Поиск в индексе:** Поиск ближайших соседей.
3. **Извлечение top-K:** Документы с наибольшим сходством.
4. **Дальнейшая обработка:** Например, **Reader** или **Re-ranker**.

### Результаты
- **Превосходство над Sparse Retrieval:** Улучшение Top-20 Accuracy на 10-20 п.п. по сравнению с BM25.
- **Эффективность Hard Negative:** Критически важны для производительности.
- **SOTA в Open-Domain QA:** Достижение state-of-the-art результатов.

<img src="img/img.png" width=500>
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример использования DPR с библиотекой Hugging Face Transformers и FAISS для индексации и поиска

from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer
import torch
import faiss
import numpy as np

# Инициализация токенайзеров и моделей для запросов и документов
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")

context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

# Пример запроса и документов
query = "What is the capital of France?"
documents = [
    "Paris is the capital of France.",
    "Berlin is the capital of Germany.",
    "Madrid is the capital of Spain."
]

# Кодирование запроса
query_inputs = question_tokenizer(query, return_tensors="pt")
query_embedding = question_encoder(**query_inputs).pooler_output.detach().numpy()

# Кодирование документов
document_embeddings = []
for doc in documents:
    context_inputs = context_tokenizer(doc, return_tensors="pt")
    doc_embedding = context_encoder(**context_inputs).pooler_output.detach().numpy()
    document_embeddings.append(doc_embedding)

document_embeddings = np.vstack(document_embeddings)

# Создание и заполнение FAISS индекса
index = faiss.IndexFlatIP(document_embeddings.shape[1])  # Используем скалярное произведение (dot-product)
index.add(document_embeddings)

# Поиск top-K документов
k = 2
distances, indices = index.search(query_embedding, k)

# Вывод результатов
print("Query:", query)
print("\nTop-K Retrieved Documents:")
for i in range(k):
    print(f"Rank {i+1}: {documents[indices[0][i]]} (Score: {distances[0][i]:.4f})")

# Ожидаемый вывод:
# Query: What is the capital of France?
#
# Top-K Retrieved Documents:
# Rank 1: Paris is the capital of France. (Score: ...)
# Rank 2: Berlin is the capital of Germany. (Score: ...)
```

### Ключевые моменты:
1. **Два энкодера**: Мы используем два отдельных энкодера BERT для запросов и документов, что позволяет им специализироваться на своих задачах.
2. **Dot-product**: Для оценки релевантности используется скалярное произведение эмбеддингов запроса и документа.
3. **FAISS**: Используется для быстрого поиска ближайших соседей, что позволяет эффективно извлекать релевантные документы из большой коллекции.
4. **Hard Negatives**: Хотя в этом примере они не показаны, в реальных сценариях обучения используются сложные негативные примеры для улучшения качества модели.

Этот пример иллюстрирует, как DPR использует специализированные энкодеры и эффективные методы поиска для решения задачи извлечения релевантных документов.